In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🧹 Data-Cleaning Approval Workflow

## What We're Building

A workflow that:
1. **Profiles a dataset** — check types, nulls, outliers, duplicates
2. **Suggests cleaning transformations** — LLM proposes fixes
3. **Pauses for user approval** — user reviews proposed changes
4. **Applies approved transformations** — executes only approved fixes

## Architecture

```
Raw Dataset
     │
     ▼
┌──────────┐    ┌────────────┐    ┌──────────┐    ┌──────────┐
│ Profile  │ ──▶│ Suggest    │ ──▶│ ⏸ PAUSE  │ ──▶│ Apply    │
│ Dataset  │    │ Fixes      │    │ Approval │    │ Fixes    │
└──────────┘    └────────────┘    └──────────┘    └──────────┘
```

## Stack
- **LangGraph** — workflow with HITL approval
- **pandas** — data profiling and transformation
- **Ollama** — LLM suggests cleaning strategies

## Step 1 — Imports & Sample Dataset

In [ ]:
import pandas as pd
import numpy as np
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)
print("All imports successful!")

In [ ]:
# Create a messy dataset with common data quality issues
np.random.seed(42)
n = 50

dirty_data = pd.DataFrame({
    "customer_id": list(range(1, n+1)),
    "name": ["Alice Smith", "Bob Jones", "  Charlie Brown ", "alice smith", "Diana Prince",
             "Eve Adams", None, "Frank Castle", "Grace Lee", "HENRY FORD"] * 5,
    "email": ["alice@email.com", "bob@email", "charlie@email.com", "alice@email.com", "diana@email.com",
              "invalid-email", "eve@email.com", None, "grace@email.com", "henry@email.com"] * 5,
    "age": [28, 35, -5, 28, 42, 150, 31, 29, np.nan, 45] * 5,
    "salary": [50000, 65000, 72000, 50000, 88000, 45000, None, 0, 91000, 55000] * 5,
    "department": ["Engineering", "engineering", "ENGINEERING", "Sales", "sales",
                   "HR", "H.R.", "Marketing", "marketing ", None] * 5,
    "join_date": ["2023-01-15", "2023/02/20", "15-Mar-2023", "2023-01-15", "2023-04-10",
                  "not_a_date", "2023-06-01", "2023-07-15", "2023-08-20", "2024-01-01"] * 5,
})

print("Dirty Dataset Sample:")
print(dirty_data.head(10).to_string())
print(f"\nShape: {dirty_data.shape}")

## Step 2 — Define State & Nodes

In [ ]:
class CleaningState(TypedDict):
    profile_report: str        # Data quality profile
    suggested_fixes: str       # LLM's cleaning suggestions
    user_approval: str         # User's approval/modifications
    cleaning_log: str          # What was actually done
    rows_before: int
    rows_after: int


# We'll keep the DataFrame as a module-level variable (not in state, since it's not serializable)
working_df = dirty_data.copy()

In [ ]:
# --- Node 1: Profile the dataset ---
def profile_dataset(state: CleaningState) -> dict:
    """Generate a data quality profile report."""
    global working_df
    df = working_df
    print("📊 Node: profile_dataset — Scanning for issues...")

    report_lines = []
    report_lines.append(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns\n")

    # Null analysis
    nulls = df.isnull().sum()
    null_cols = nulls[nulls > 0]
    if len(null_cols) > 0:
        report_lines.append("MISSING VALUES:")
        for col, count in null_cols.items():
            pct = count / len(df) * 100
            report_lines.append(f"  - {col}: {count} nulls ({pct:.0f}%)")

    # Duplicate check
    dupes = df.duplicated().sum()
    if dupes > 0:
        report_lines.append(f"\nDUPLICATES: {dupes} exact duplicate rows")

    # Numeric outliers
    report_lines.append("\nNUMERIC COLUMNS:")
    for col in df.select_dtypes(include=[np.number]).columns:
        stats = df[col].describe()
        report_lines.append(f"  - {col}: min={stats['min']}, max={stats['max']}, mean={stats['mean']:.1f}")
        # Flag obvious issues
        if col == 'age':
            bad = df[(df[col] < 0) | (df[col] > 120)][col].dropna()
            if len(bad) > 0:
                report_lines.append(f"    ⚠️ {len(bad)} values outside valid range (0-120): {bad.unique().tolist()}")
        if col == 'salary' and (df[col] == 0).any():
            report_lines.append(f"    ⚠️ {(df[col] == 0).sum()} zero-value salaries")

    # String inconsistencies
    report_lines.append("\nSTRING COLUMNS:")
    for col in df.select_dtypes(include=['object']).columns:
        unique = df[col].dropna().nunique()
        report_lines.append(f"  - {col}: {unique} unique values")
        # Check for whitespace/case issues
        stripped = df[col].dropna().str.strip().str.lower()
        unique_normalized = stripped.nunique()
        if unique_normalized < unique:
            report_lines.append(f"    ⚠️ {unique - unique_normalized} values differ only by case/whitespace")

    profile = "\n".join(report_lines)
    print(f"     Found {profile.count('⚠️')} issues")
    return {"profile_report": profile, "rows_before": len(df)}


# --- Node 2: Suggest fixes ---
suggest_prompt = ChatPromptTemplate.from_template(
    """You are a data engineer. Based on this data quality profile,
suggest specific cleaning transformations.

Data Profile:
{profile}

For each issue, provide:
- FIX #N: Short name
- ISSUE: What's wrong
- ACTION: Specific transformation to apply
- RISK: low/medium/high (could we lose valid data?)

Suggested fixes:"""
)


def suggest_fixes(state: CleaningState) -> dict:
    print("💡 Node: suggest_fixes — Generating recommendations...")
    chain = suggest_prompt | llm | StrOutputParser()
    fixes = chain.invoke({"profile": state["profile_report"]})
    return {"suggested_fixes": fixes}


# --- Node 3: Apply approved fixes ---
def apply_fixes(state: CleaningState) -> dict:
    """Apply the cleaning transformations the user approved."""
    global working_df
    df = working_df
    print("🔧 Node: apply_fixes — Applying approved transformations...")

    log = []
    approval = state["user_approval"].lower()

    # Apply common fixes based on what user approved
    # Strip whitespace from string columns
    if "whitespace" in approval or "approve all" in approval or "strip" in approval:
        for col in df.select_dtypes(include=['object']).columns:
            df[col] = df[col].str.strip() if df[col].dtype == 'object' else df[col]
        log.append("✓ Stripped whitespace from all string columns")

    # Standardize case in department
    if "case" in approval or "approve all" in approval or "department" in approval:
        df["department"] = df["department"].str.strip().str.title()
        df["department"] = df["department"].replace({"H.R.": "Hr", "Hr": "HR"})
        log.append("✓ Standardized department names to title case")

    # Fix invalid ages
    if "age" in approval or "approve all" in approval or "outlier" in approval:
        bad_ages = (df["age"] < 0) | (df["age"] > 120)
        df.loc[bad_ages, "age"] = np.nan
        log.append(f"✓ Set {bad_ages.sum()} invalid ages to NaN")

    # Remove exact duplicates
    if "duplicate" in approval or "approve all" in approval:
        before = len(df)
        df = df.drop_duplicates()
        log.append(f"✓ Removed {before - len(df)} duplicate rows")

    # Fill null salaries with median
    if "salary" in approval or "approve all" in approval or "null" in approval:
        median_sal = df["salary"].median()
        null_count = df["salary"].isnull().sum()
        df["salary"] = df["salary"].fillna(median_sal)
        df.loc[df["salary"] == 0, "salary"] = median_sal
        log.append(f"✓ Filled {null_count} null + zero salaries with median ({median_sal})")

    working_df = df
    cleaning_log = "\n".join(log) if log else "No fixes applied (user did not approve any)"
    print(f"     Applied {len(log)} transformations")
    return {"cleaning_log": cleaning_log, "rows_after": len(df)}


print("All nodes defined")

## Step 3 — Build Graph with Approval Gate

In [ ]:
workflow = StateGraph(CleaningState)

workflow.add_node("profile_dataset", profile_dataset)
workflow.add_node("suggest_fixes", suggest_fixes)
workflow.add_node("apply_fixes", apply_fixes)

workflow.set_entry_point("profile_dataset")
workflow.add_edge("profile_dataset", "suggest_fixes")
workflow.add_edge("suggest_fixes", "apply_fixes")
workflow.add_edge("apply_fixes", END)

memory = MemorySaver()
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["apply_fixes"]  # Pause before applying changes
)

print("Data cleaning workflow compiled (pauses before applying fixes)")

## Step 4 — Run: Profile & Suggest

In [ ]:
config = {"configurable": {"thread_id": "cleaning-001"}}

result = app.invoke({
    "profile_report": "",
    "suggested_fixes": "",
    "user_approval": "",
    "cleaning_log": "",
    "rows_before": 0,
    "rows_after": 0,
}, config)

print("\n" + "="*60)
print("📊 DATA QUALITY PROFILE")
print("="*60)
print(result["profile_report"])

print("\n" + "="*60)
print("💡 SUGGESTED FIXES (review before approving)")
print("="*60)
print(result["suggested_fixes"])

print("\n⏸️ Workflow paused — waiting for your approval...")

## Step 5 — User Approves & Apply

In [ ]:
# User approves all suggested fixes
user_approval = """
Approve all fixes:
- Strip whitespace from all columns
- Standardize department case
- Fix invalid age outliers (set to NaN)
- Remove duplicate rows
- Fill null/zero salary with median
"""

app.update_state(config, {"user_approval": user_approval})

print("🔄 Applying approved fixes...\n")
final = app.invoke(None, config)

print("\n" + "="*60)
print("✅ CLEANING COMPLETE")
print("="*60)
print(final["cleaning_log"])
print(f"\nRows: {final['rows_before']} → {final['rows_after']}")

print("\n📋 Cleaned data sample:")
print(working_df.head(10).to_string())

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Data profiling** | Automated quality checks (nulls, outliers, dupes) |
| **LLM suggestions** | LLM interprets profile and proposes fixes |
| **Approval gate** | `interrupt_before` pauses for user review |
| **Safe execution** | Only approved transformations are applied |

## Why HITL for Data Cleaning?

- Prevents accidental data loss from aggressive cleaning
- User sees *what will change* before it happens
- Creates an audit trail of cleaning decisions

## 🔧 Extensions

- **Preview mode**: Show before/after for each fix before applying
- **Rollback**: Save original data and allow undo
- **Schema validation**: Verify cleaned data matches expected schema
- **Real pandas codegen**: Have LLM generate actual `.py` code, not just descriptions